### 4. Building height calculation.

These notebook takes the output from notebook [No.2](https://10.243.64.5/user/julian/lab/tree/handover/notebooks/2_filter_and_extract_buildings_from_S2_Partitions.ipynb), where some basic parameters are computed and the geotif files that we obtained from the [previous notebook](https://10.243.64.5/user/julian/lab/tree/handover/notebooks/3_download_google_25D.ipynb), are used to compute the height per building. The main function 'calculate_heights_in_tiff' computes building height statistics by cross-referencing building footprints with a corresponding elevation raster. It processes the raster in predefined tiles to handle potentially large files, calculates height metrics for each building within a tile, and returns a single, enriched DataFrame with the new height information.

### Initial configuration

    config_str = """{"REGION": "Patan",
    "COS_ENDPOINT_URL": "https://s3.direct.eu-de.cloud-object-storage.appdomain.cloud",
    "COUNTRY_NAME": "India",
    "FILE_PREFIX": "Patan",
    "VIDA_PARQUET_BUCKET": "parquets",
    "VIDA_S2_PARTITIONS": "onboarding-bucket-j",
    "COS_AUTH_ENDPOINT_URL":"https://iam.cloud.ibm.com/oidc/token",
    "COS_APIKEY": "xxx",
    "UTILS_BUCKET": "notebook-utils-bucket",
    "VIDA_COUNTRIES_BUILDINGS": "onboarding-bucket-j"}"""

* `REGION`: Region of study.
* `COS_ENDPOINT_URL`: The S3 API endpoint for your IBM COS instance.
* `COUNTRY_NAME`: Defines the target country to filter the google dataset.
* `VIDA_PARQUET_BUCKET`: The name of the source bucket where the Parquet files are stored.
* `COS_AUTH_ENDPOINT_URL`: The authentication endpoint for generating IBM IAM tokens.
* `COS_APIKEY`: Your unique IBM Cloud API key for programmatic access.
* `UTILS_BUCKET`: Where the .py file to download the package `filtering_grid_generator`.
* `VIDA_COUNTRIES_BUILDINGS`: Bucket to upload results.

In [7]:
# Import necessary libraries
import io
#from PIL import Image
import ibm_boto3
# import jaydebeapi as jdbc
# import jpype
from botocore.client import Config
import numpy as np
#import configparser
import os
import sys
from ibm_cloud_sdk_core import ApiException
from ibmcloudant.cloudant_v1 import CloudantV1, Document, BulkDocs
from ibm_cloud_sdk_core.authenticators import IAMAuthenticator
import pandas as pd
import geopandas as gpd
#import random
import time
import base64
import shutil
import threading
from collections import Counter
from tqdm import tqdm
from datetime import datetime
import rasterio
from rasterio.windows import Window
import gc
from matplotlib.path import Path
import matplotlib.pyplot as plt
import shapely
from rasterio.plot import show
import rioxarray
import traceback
from datetime import datetime
from glob import glob
import getpass
import json

In [3]:
datetime.now().strftime("%Y-%m-%d %H:%M:%S")

'2025-10-03 09:19:56'

In [4]:
#config_str = getpass.getpass('Enter your prepared config: ')
config = json.loads(config_str)

In `output_folder` results will be temporary saved. 

In [5]:
output_folder = "/home/julian/notebooks/"

The next cell dynamically downloads and imports a custom Python utility module `utils.py` and its configuration data (.json) from IBM Cloud Object Storage (COS).

In [8]:
# init S3 client in order to work with last tiff file version
cos_client = ibm_boto3.client(service_name='s3',
                              ibm_api_key_id=config["COS_APIKEY"],
                              config=Config(signature_version='oauth'),
                              endpoint_url=config["COS_ENDPOINT_URL"])


# import external utils library
response = cos_client.list_objects_v2(Bucket=config["UTILS_BUCKET"])

utils_to_download = ['utils.py', 'S2_MGRS_tile_names.json']

try:
    for obj in response['Contents']:
        name = obj['Key']

        if name in utils_to_download:
            streaming_body_1 = cos_client.get_object(Bucket=config["UTILS_BUCKET"], Key=name)['Body']
            print("Copying to localStorage :  " + name)
            with io.FileIO(name, 'w') as file:
                for i in io.BytesIO(streaming_body_1.read()):
                    file.write(i)
        
    from utils import *
    print('External utils succesfully imported')
except Exception as e:
    print('Error occured: ', e)

Copying to localStorage :  S2_MGRS_tile_names.json
Copying to localStorage :  utils.py
External utils succesfully imported


In [9]:
path_to_tif_folder = 'tiffs'  #temporal folder where tiffs would be downloaded

In [10]:
# create a resource to be able to retrieve all the object in the bucket
cos_client_resource = ibm_boto3.resource(service_name='s3',
                              ibm_api_key_id=config["COS_APIKEY"],
                              config=Config(signature_version='oauth'),
                              endpoint_url=config["COS_ENDPOINT_URL"])
# download work parquet file
response=cos_client.list_objects_v2(Bucket=config["VIDA_COUNTRIES_BUILDINGS"])

# create a bucket instance
parquet_bucket = cos_client_resource.Bucket(config["VIDA_COUNTRIES_BUILDINGS"])

# get all filenames from the parquet bucket
all_parquet_files = [i.key for i in parquet_bucket.objects.all()]

country_parquet_filename =\
[i for i in all_parquet_files if (config['REGION']+'_buildings' in i) and ('.parquet' in i)] #looking for file from previous notebook


In [11]:
#all_parquet_files
parquet_to_download = country_parquet_filename

try:
    for obj in response['Contents']:
        name = obj['Key']

        if name in parquet_to_download:
            streaming_body_1 = cos_client.get_object(Bucket=config["VIDA_COUNTRIES_BUILDINGS"], Key=name)['Body']
            print("Copying to localStorage :  " + name)
            with io.FileIO(name, 'w') as file:
                for i in io.BytesIO(streaming_body_1.read()):
                    file.write(i)

except Exception as e:
    print('Error occured: ', e)

Copying to localStorage :  /home/julian/notebooks/Patan_buildings.parquet


Reading the downloaded file to fetch in building heighst and its lenght for postprocessing.

In [12]:
main_df = pd.read_parquet(country_parquet_filename)
main_df_len = len(main_df)

In [13]:
# create a resource to be able to retrieve all the object in the bucket
cos_client_resource = ibm_boto3.resource(service_name='s3',
                              ibm_api_key_id=config["COS_APIKEY"],
                              config=Config(signature_version='oauth'),
                              endpoint_url=config["COS_ENDPOINT_URL"])

# create a bucket instance
tiffs_bucket = cos_client_resource.Bucket(config["BUCKET_TIFF"])

# get all filenames from the tiffs bucket
all_tiff_files = [i.key for i in tiffs_bucket.objects.all()]
state = config["REGION"]
country_tile_filenames = [i for i in all_tiff_files if (state in i) and ('.tif' in i)] 

All of the coming functions arre embebed into the main one Calculating_heights_in_tiff:
* `reproject_tif_CRS` It reprojects a GeoTIFF file to the standard WGS 84 latitude/longitude coordinate system and overwrites the original file with the result.It is memory intensive, if the infrastructure where is getting run, can't handle large datasets, it is necessary to break the tiffs into smaller rasters using the function: *tiling*
* `generate_imagin_coords`: This function divides a large raster into a grid of smaller, overlapping tiles and returns a list of pixel coordinates that define the boundaries for each of these tiles.
* `fetch_buildings_in_bbox`*: It filters a pandas DataFrame to find all rows (buildings) that fall within a specified rectangular geographic area (a bounding box).
* `get_min_max_values_row_col`: it calculates the bounding box of a list of pixel coordinates by finding the minimum and maximum row and column values.
* `download_tiff`: This function handles both the download and a key pre-processing step for raster data. It ensures the resulting file is a single-band GeoTIFF (band No 2.)
* `categorize_height`: It snaps the height dataset to the nearest logical:
    * If the height is NaN (not a number), it defaults to 4.5 meters.
    * Any height up to 4.5m is categorized as a single-story building and standardized to 4.5m.
    * Any height between 4.5m and 7.5m is categorized as a two-story building and standardized to 7.5m.
    * For any height above 7.5m, it rounds the height up to the next 3-meter increment (e.g., 8.5m becomes 10.5m, 11m becomes 13.5m, etc.)
* `calculate_floors_gfa` This function takes the clean, standardized height from the previous function, along with the building's footprint area, and calculates the final number of floors and the total GFA.

In [15]:
def reproject_tif_CRS(filename: str):
    rds = rioxarray.open_rasterio(filename)
    rds_4326 = rds.rio.reproject("EPSG:4326")
    rds_4326.rio.to_raster(filename, compress="DEFLATE")
    
def generate_image_coords(heights_tiff_name:str, div_arg:int = 1) -> list:
    
    dat = rasterio.open(os.path.join(path_to_tif_folder, heights_tiff_name))
    profile = dat.profile.copy()
    profile.update(compress='lzw')
    # print(profile)
    
    #divide tiff to tiles
    tiff_width = profile['width']
    tiff_height = profile['height']
    
    tile_width = int(tiff_width / div_arg)
    tile_height = int(tiff_height / div_arg)
    
    print(f'tile_width: {tile_width}, tile_height: {tile_height}')
    # define overlap between tiles
    overlap = 1000
    
    columns_amount = int(tiff_width / tile_width) if tiff_width % tile_width == 0 else int(tiff_width / tile_width) + 1
    rows_amount = int(tiff_height / tile_height) if tiff_height % tile_height == 0 else int(tiff_height / tile_height) + 1
    print(f'TIFf image wiil be divided to {rows_amount} rows and {columns_amount} cols')
    
    images_coords = []
    
    for col_idx in range(1, columns_amount + 1):
        
        row_start = max(tile_width * (col_idx - 1) - overlap, 0)
        
        if col_idx != columns_amount:
            
            row_limits = [row_start, tile_width * col_idx]
        elif col_idx == columns_amount:
            row_limits = [row_start, tiff_width]
        
        for row_idx in range(1, rows_amount + 1):
            
            col_start = max(tile_height * (row_idx - 1) - overlap, 0)
            
            if row_idx != columns_amount:
                col_limits = [col_start, tile_height * row_idx]
            elif row_idx == columns_amount:
                col_limits = [col_start, tiff_height]
                
            coords = [col_limits, row_limits]
            images_coords.append(coords)

    return images_coords

fetch_builings_in_bbox =\
lambda df, lon_min, lon_max, lat_min, lat_max:\
df[(df.longitude >= lon_min) & (df.longitude <= lon_max) & (df.latitude >= lat_min) & (df.latitude <= lat_max)]

def get_min_max_values_row_col(pixel_coordinates: list):
    
    return {
        'rowminmax': 
            [
                min([i[0] for i in pixel_coordinates]), 
                max([i[0] for i in pixel_coordinates]), 
            ], 
        'colminmax': 
            [
                min([i[1] for i in pixel_coordinates]), 
                max([i[1] for i in pixel_coordinates]), 
            ]
            }
def download_tiff(path_to_tif_folder, heights_tiff_name):
    
    try:
        streaming_body_1 = cos_client.get_object(Bucket=config["BUCKET_TIFF"], Key=heights_tiff_name)['Body']
        print("Downloading " + heights_tiff_name)

        os.makedirs(path_to_tif_folder, exist_ok=True)
        result_path = os.path.join(path_to_tif_folder, heights_tiff_name)
        with io.FileIO(result_path, 'w') as file:
            for i in io.BytesIO(streaming_body_1.read()):
                file.write(i)

        with rasterio.open(result_path) as src:
            out_meta = src.meta
            data = src.read(2)

            out_meta['count'] = 1
            
            with rasterio.open(result_path, "w", **out_meta) as dest:
                dest.write(data, 1)

        return result_path
        
    except Exception as e:
        print(f"Downloading exception occurred: {e}")
        return

def calculate_floors_gfa(height_categorized, area_in_meters):

    if np.isnan(height_categorized):
        height_categorized = 1
        
    def get_floor(height):
        if (height >= 0) and (height <= 4.5):
            return 1
        elif (height > 4.5) and (height <= 7.5):
            return 2
        elif (height > 7.5):
            return int(((height - 1.5005) // 3 * 3 + 3) / 3)

    floors = get_floor(height_categorized)
    gfa = round(area_in_meters * floors, 5)

    return floors, gfa


def categorize_height(height):
    if np.isnan(height):
        return 4.5
    height = round(height, 5)
    if (height >= 0) & (height <= 4.5):
        return 4.5
    elif (height > 4.5) & (height <= 7.5):
        return 7.5
    else:
        return (int(int(height - 4.5) // 3) * 3) + 7.5

json_filename = f"Height_calculation_{config['REGION']}.json"

def log_state_to_bucket(inferenced_count: dict):
    
    with open(json_filename, "w") as outfile:
                json.dump(inferenced_count, outfile)
                
    cos_client.upload_file(
        Filename=json_filename,
        Bucket='notebook-job-status',
        Key=json_filename,
        )

The `calculate_heights_in_tiff` function is a geospatial processing tool designed to enrich a DataFrame of building footprints (main_df) with height information derived from a GeoTIFF file. To manage potentially large files, it first calls a helper function, `generate_image_coords`, which creates a plan to process the raster in smaller, overlapping chunks or "tiles." The main function then iterates through each of these tiles, first finding all the buildings from the master DataFrame that fall within the tile's geographic area. It then reads only that small corresponding chunk of the raster's pixel data into memory. For each building within the chunk, it creates a precise pixel mask of its footprint, uses the mask to extract the underlying elevation values, and calculates statistics such as the median and mean height. These values are then used by other helper functions to estimate and add new columns to the DataFrame. The final output is a single DataFrame which contains the new attributes: number of floors, gross floor area and height.

In [16]:
def calculate_heights_in_tiff(main_df, heights_tiff_name):

    t1 = time.time()
    tiff_path = download_tiff(path_to_tif_folder, heights_tiff_name) 

    if tiff_path != None:
        print('reproject_tif_CRS')
        reproject_tif_CRS(tiff_path)

    images_coords = generate_image_coords(heights_tiff_name)
    dfs = []
    
    
    # loop through tiles coords
    for idx, coords in enumerate(images_coords):


        with rasterio.open(tiff_path) as src:
            profile = src.profile.copy()
            tiff_width = profile['width']
            tiff_height = profile['height']
            # read tiff metadata by coords in order to prepare filtered dataframe
            print(coords)
            
            col_off = coords[1][0]
            row_off = coords[0][0]
            
            width = coords[1][1] - coords[1][0]
            height = coords[0][1] - coords[0][0]
            
            
            lon_upper_left, lat_upper_left = src.xy(coords[0][0], coords[1][0])
            lon_down_right, lat_down_right = src.xy(coords[0][1], coords[1][1])
    
            lons_sorted = sorted([lon_upper_left, lon_down_right])
            lats_sorted = sorted([lat_upper_left, lat_down_right])
            
            lon_min = lons_sorted[0]
            lon_max = lons_sorted[1]
    
            lat_min = lats_sorted[0]
            lat_max = lats_sorted[1]
            
            areas_covered_by_tifs = create_bounds_dict(path_to_tifs=path_to_tif_folder)
    
            # set up default height (3m - 1 floor) for buildings under the threshold (20 square meters)
            # upd_default_height_in_bbox(lon_min, lon_max, lat_min, lat_max)
    
            # fetch all buildings larger than the threshold (20 square meters) to estimate its height
            df = fetch_builings_in_bbox(main_df, lon_min, lon_max, lat_min, lat_max).copy()
            df['geometry'] = df['geometry'].apply(shapely.from_wkb)
            init_len = len(df)
            # coordinates of current tile
            col_off = max(coords[1][0] - 1, 0)
            row_off = max(coords[0][0] - 1, 0)
            width = min(coords[1][1] - coords[1][0] + 1, tiff_width)
            height = min(coords[0][1] - coords[0][0] + 1, tiff_height)
            
            print('offsets', col_off, row_off)
            
            # read tile grayscale layer
            tiff_data = src.read(1, window=Window(col_off, row_off, width, height))
            
            tiff_data[tiff_data == -99.0] = np.nan
            
            print(f"Images revealed: {len(df)}")
            
            # loop through building centroids inside tile

            df.index = [i for i in range(len(df))]
            
            for index, row, in tqdm(df.iterrows(), total=len(df), desc='Height calculation'):
                try:
    
                    # lat lon to pixel transformations
                    pixel_coordinates = get_pixel_coordinates(row.geometry, areas_covered_by_tifs, src)
                    polygon_coordinates = [[pixel_coords[0] - row_off, pixel_coords[1] - col_off] for pixel_coords in pixel_coordinates]
                    
                    margin = 0
                    
                    rowcolminmax = get_min_max_values_row_col(polygon_coordinates)
                    
                    img_width = rowcolminmax['rowminmax'][1] - rowcolminmax['rowminmax'][0] 
                    img_height = rowcolminmax['colminmax'][1] - rowcolminmax['colminmax'][0]
                    
                    row_start = rowcolminmax['rowminmax'][0]
                    row_end = rowcolminmax['rowminmax'][1]
                    col_start = rowcolminmax['colminmax'][0]
                    col_end = rowcolminmax['colminmax'][1]

                    
                    # img_array_pre = np.array(tiff_data[row_start : row_end, col_start : col_end])
                    
    
                    # polygon_coordinates = offset_polygon_coords(polygon_coordinates)
                    # rowcolminmax = get_min_max_values_of_row_col(polygon_coordinates)
                    
                    # img_width = rowcolminmax['rowminmax'][1] - rowcolminmax['rowminmax'][0] 
                    # img_height = rowcolminmax['colminmax'][1] - rowcolminmax['colminmax'][0]
                    # row_start = rowcolminmax['rowminmax'][0]
                    # row_end = rowcolminmax['rowminmax'][1]
                    # col_start = rowcolminmax['colminmax'][0]
                    # col_end = rowcolminmax['colminmax'][1]

                    # cut building image from tile
                    img_array_pre = np.array(tiff_data[row_start : row_end, col_start : col_end])

                    if img_array_pre.shape != (img_width, img_height):
                        print('building polygon out of tiff')
                        np_nan_matrix = np.empty((img_width, img_height))
                        np_nan_matrix.fill(np.nan)
                        np_nan_matrix[:img_array_pre.shape[0], :img_array_pre.shape[1]] = img_array_pre
                        img_array_pre = np_nan_matrix
                    
                    # extract building by polygon coords
                    absolule_polygon_coordinates = [[pixel_coords[0] - row_start, pixel_coords[1] - col_start] for pixel_coords in polygon_coordinates]
                    poly_path=Path(absolule_polygon_coordinates)
                    x, y = np.mgrid[:img_height, :img_width]
                    coors = np.hstack((x.reshape(-1, 1), y.reshape(-1,1)))
                    mask = poly_path.contains_points(coors).reshape(img_height, img_width).T
                    
                    # create zeros mask
                    img_masked=np.zeros((img_width, img_height),dtype=img_array_pre.dtype)
    
                    # put image on zeros mask
                    img_masked[mask]=img_array_pre[mask]
    
                    # extract image as list of non zero values
                    
                    img_masked_list = list(filter(lambda num: num != 0, img_masked.flatten(order='C')))

                    if len(img_masked_list) > 0:
                        img_masked_list = [i for i in img_masked_list if not np.isnan(i)]
                        
                        if len(img_masked_list) == 0:
                            img_masked_list = np.array([0])
                    else:
                        img_masked_list = np.array([0])
                        
                    if len(img_masked_list) > 0:    
                        nanmedian_height = np.nanmedian(img_masked_list)
                        
                        df.at[index, 'height_mean'] = np.nanmean(img_masked_list)
                        df.at[index, 'height_median'] = nanmedian_height
                        df.at[index, 'height_max'] = np.max(img_masked_list)
                        df.at[index, 'height'] = categorize_height(nanmedian_height)
    
                        floors, gfa = calculate_floors_gfa(nanmedian_height, row.area_in_meters)
                        df.at[index, 'floors'] = floors
                        df.at[index, 'gfa_in_meters'] = gfa
                        
                except Exception as e:
                    # pass
                    print(f'Height calculation error {e}')
                    traceback.print_exc()
                
            dfs.append(df)
            
    try:
        result_df = pd.concat(dfs)

        result_len = len(result_df)
        print(f'Init len {init_len} -> result len {result_len} | diff: {init_len - result_len}')

        print('Remove', tiff_path)
        os.remove(tiff_path)
        return result_df

    except Exception as e:
        print(f'Concat or upload error occurred: {e}')


The next part of the code iterates throug each tiff file and computes the calculate_heights_in_tiff function. The results are stored in parquet files with the prefix "lost_part_".

In [17]:
total_count = 0

for tidx, tiff_name in enumerate(country_tile_filenames):
# -1 to start from beginning
    if tidx > -1:
        tile_progress = f'Processing {tiff_name}  {tidx+1} of {len(country_tile_filenames)}'
        print(tile_progress)
    
        heights_df = calculate_heights_in_tiff(main_df, tiff_name)
        total_count += len(heights_df)
        
        if len(heights_df) > 0:
            try:
                filename = tiff_name.replace('.tif', '.parquet')
                filename = f'lost_part_{filename}'
                heights_df = gpd.GeoDataFrame(heights_df, geometry=heights_df.geometry)
                heights_df.to_parquet(filename)
                res=cos_client.upload_file(Filename=filename, Bucket=config["HEIGHT_BUCKET"],Key=filename)
                parquet_upload_status = f'Parquet: {filename} uploaded'
                os.remove(filename)
            except Exception as e:
                parquet_upload_status = f'Parquet: {filename} not uploaded error: {e}'
    
            state = dict(
                calculated_count = total_count,
                progress = f'{100*round(total_count/main_df_len, 6)}% | {total_count} of {main_df_len}',
                parquet_upload_status = parquet_upload_status,
                tile = tile_progress
            )
        
            log_state_to_bucket(state)
            print()
        
        

Processing Patan_tile_-N_uyIhuU1w.tif  1 of 23
reproject_tif_CRS
tile_width: 26529, tile_height: 24355
TIFf image wiil be divided to 1 rows and 1 cols
[[0, 24355], [0, 26529]]
offsets 0 0
Images revealed: 6211


Height calculation:   1%|          | 59/6211 [00:00<00:10, 584.17it/s]

building polygon out of tiff
building polygon out of tiff


Height calculation:   4%|▍         | 253/6211 [00:00<00:09, 627.89it/s]

building polygon out of tiff
building polygon out of tiff


...

building polygon out of tiff
building polygon out of tiff
building polygon out of tiff
building polygon out of tiff
building polygon out of tiff
building polygon out of tiff
building polygon out of tiff


Height calculation:  36%|███▌      | 7309/20557 [00:09<00:16, 819.56it/s]

building polygon out of tiff


Height calculation: 100%|██████████| 20557/20557 [00:26<00:00, 774.58it/s]


Init len 20557 -> result len 20557 | diff: 0
Remove tiffs/Patan_tile_wu8TazXhUDk.tif



### Merge parquets for further operations

This code connects to an IBM Cloud Object Storage (COS) bucket, gets a list of all the files inside, and then filters that list to find specific Parquet files related to a particular region with the prefix `lost_part_` and `parquet` in the name. 

In [18]:
# create a resource to be able to retrieve all the object in the bucket
cos_client = ibm_boto3.client(service_name='s3',
                              ibm_api_key_id=config["COS_APIKEY"],
                              config=Config(signature_version='oauth'),
                              endpoint_url=config["COS_ENDPOINT_URL"])

# Connect to COS as a resource (not client!)
cos_resource = ibm_boto3.resource(service_name='s3',
                                  ibm_api_key_id=config["COS_APIKEY"],
                                  config=Config(signature_version='oauth'),
                                  endpoint_url=config["COS_ENDPOINT_URL"])

parquet_bucket = cos_resource.Bucket(config["HEIGHT_BUCKET"])

# get all filenames from the parquet bucket
all_parquet_files = [i.key for i in parquet_bucket.objects.all()]

state = config["REGION"]
parquets_to_merge = [i for i in all_parquet_files if ('lost_part_'+state in i) and ('.parquet' in i)] 

This will download a specific list of Parquet files from an IBM Cloud Object Storage (COS) bucket to a local directory on your machine.

In [30]:
# Get REGION related parquets
local_dir = os.path.join(output_folder,"downloaded_parquets")
os.makedirs(local_dir, exist_ok=True)

for file_key in parquets_to_merge:
    local_path = os.path.join(local_dir, os.path.basename(file_key))
    print(f"Downloading {file_key} to {local_path}...")
    
    try:
        cos_client.download_file(
            Bucket=config["HEIGHT_BUCKET"],
            Key=file_key,
            Filename=local_path
        )
    except Exception as e:
        print(f"Failed to download {file_key}: {e}")

All of the files are merged in the next cell. For that all files that were gotten from the IBM bucket with the name "lost_part_RegionsName" are temporaly saved in a folder. Then all of them are merged in a single parquet file, duplicates are deleted and final parquet file is uploaded in a bucket.

In [31]:
# Define path to folder with downloaded Parquet files
parquet_folder = os.path.join(output_folder,"downloaded_parquets")
os.makedirs(parquet_folder, exist_ok=True)

# Get all .parquet files in the folder
parquet_files = glob(os.path.join(parquet_folder, "*.parquet"))

# Read and concatenate all Parquet files
df_list = [pd.read_parquet(file) for file in parquet_files]
merged_df = pd.concat(df_list, ignore_index=True)

# Drop exact duplicates
merged_df.drop_duplicates(inplace=True)
merged_df.drop_duplicates(subset=["geometry"], inplace=True)


# Optional: drop duplicates by specific key columns
# merged_df.drop_duplicates(subset=["geometry", "boundary_id"], inplace=True)

new_file_name = os.path.join(output_folder, state + "_height_NEW.parquet")

# Save the merged dataset to a new Parquet file
merged_df.to_parquet(new_file_name, index=False)

print(f"Merged {len(parquet_files)} files into '{state}_height.parquet' with {len(merged_df)} unique rows.")

try:
    shutil.rmtree(parquet_folder)
    print(f"Successfully deleted {parquet_folder}.")
except Exception as e:
    print("Failed to delete folder:", e)

merged_filename = os.path.join(output_folder,state + "_height_NEW.parquet")

# Upload to COS
try:
    cos_client.upload_file(
        Filename=merged_filename,
        Bucket=config["HEIGHT_BUCKET"],
        Key=f"{merged_filename}"  # or just merged_filename if you want it in the root
    )
    print(f"Successfully uploaded {merged_filename} to COS {config["HEIGHT_BUCKET"]}.")
except Exception as e:
    print("Upload failed:", e)


Merged 23 files into 'Patan_height.parquet' with 182313 unique rows.
Successfully deleted /home/julian/notebooks/downloaded_parquets.
Successfully uploaded /home/julian/notebooks/Patan_height_NEW.parquet to COS onboarding-bucket-j.


In [21]:
datetime.now().strftime("%Y-%m-%d %H:%M:%S")

'2025-10-03 09:45:17'